In [0]:
# ==================================================================
# NOTEBOOK: nb_silver_products
# PURPOSE:  Full refresh Bronze → Silver for Products
# RUN:      Weekly (every Sunday after ADF full refresh)
#           Also run ONCE manually now to seed Silver
# SOURCE:   bronze/products/  (parquet from ADF)
# TARGET:   silver/products/  (Delta format)
# 300 products
# =================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, upper,trim, initcap,to_date, when, round, datediff, current_date


BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/products/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/products/"


# ── READ Bronze ───────────────────────────────────────────────
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[DONE] Rows read: {bronze_df.count} rows")
bronze_df.printSchema()


# ── STEP 1: Deduplication on ProductID ───────────────────────

before = bronze_df.count()
bronze_df = bronze_df.dropDuplicates(["ProductID"])
after = bronze_df.count()

if before != after:
    print(f"[WARN] Removed:{before - after} duplicate ProductID's")
print(f"[CLEAN] After dedup: {after} rows")

# ── STEP 2: Remove nulls on important/keys columns ──────────────────

bronze_df = (
    bronze_df
    .filter(col("ProductID").isNotNull())
    .filter(col("ProductName").isNotNull())
    .filter(col("ListPrice").isNotNull())
    .filter(col("ListPrice").cast("double") > 0)
)

# ── STEP 3: Type casting ──────────────────────────────────────

silver_df = (
    bronze_df
    .withColumn("ListPrice",    col("ListPrice").cast("decimal(10,2)"))
    .withColumn("CostPrice",    col("CostPrice").cast("decimal(10,2)"))
    .withColumn("StockQuantity",   col("StockQuantity").cast("integer"))
    .withColumn("Rating",        col("Rating").cast("decimal(3,1)"))
    .withColumn("LaunchDate",       to_date("LaunchDate"))
    .withColumn("LastModifiedDate",  to_date("LastModifiedDate"))
    )

# ── STEP 4: Standardize string columns ───────────────────────

silver_df = (
    silver_df
    .withColumn("ProductID",     upper(trim(col("ProductID"))))
    .withColumn("SellerID",      upper(trim(col("SellerID"))))
    .withColumn("ProductName",    initcap(trim(col("ProductName"))))
    .withColumn("Category",    initcap(trim(col("Category"))))
    .withColumn("SubCategory",     initcap(trim(col("SubCategory"))))
    .withColumn("Brand",        initcap(trim(col("Brand"))))
    .withColumn("IsActive",      upper(trim(col("IsActive"))))
)

# ── STEP 5: Business derived columns ─────────────────────────
    # Is product currently active?
silver_df = (
    silver_df
    .withColumn("IsActiveBool",  col("IsActive") == "TRUE")
    

# ── MARGIN ANALYSIS ───────────────────────────────────────
    
# Gross Margin = how much profit as % of selling price
# Formula: (ListPrice - CostPrice) / ListPrice × 100
    
    .withColumn("GrossMarginAmt", round(col("ListPrice") - col("CostPrice"), 2))
    .withColumn("GrossMarginPct",
        when(col("ListPrice") > 0,
             round(((col("ListPrice") - col("CostPrice")) / col("ListPrice")) * 100, 2))
        .otherwise(0.0))
    
# Discount = how much discount as % of selling price
# Formula: (ListPrice - SalePrice) / ListPrice × 100
    

# ── PRICE BAND ────────────────────────────────────────────
# Segment products by price for easier filtering in reports
#   
    .withColumn("PriceBand",     
                when(col("ListPrice") < 500, "BUDGET")   # under ₹500
                .when(col("ListPrice") < 2000, "MID_RANGE")    # ₹500–₹2000
                .when(col("ListPrice") < 10000, "PREMIUM")    # ₹2000–₹10000
                .otherwise("LUXURY"))                          # above ₹10000 

# ── STOCK STATUS ──────────────────────────────────────────


    .withColumn("StockStatus",   
                when(col("StockQuantity") == 0, "OUT_OF_STOCK")
                .when(col("StockQuantity") <= 10, "LOW_STOCK")
                .when(col("StockQuantity") <=50, "MEDIUM_STOCK")
                .otherwise("IN_STOCK"))
    .withColumn("IsInStock", col("StockQuantity") > 0)
    


# ── PRODUCT AGE ───────────────────────────────────────────         
# Days since product was launched on platform
 
    .withColumn("DaysSinceLaunch", datediff(current_date(), col("LaunchDate")))

     # Is this a new product? (launched in last 90 days)

     .withColumn("IsNewProduct",    
                when(col("DaysSinceLaunch")  <= 90, "NEW")
                .when(col("DaysSinceLaunch") <=365, "RECENT")
                .when(col("DaysSinceLaunch") <= 1095, "ESTABLISHED") # 3 years
                .otherwise("MATURE")
        )

# ── RATING BAND ───────────────────────────────────────────
        .withColumn("RatingBand",
                    when(col("Rating") >= 4.5,  "EXCELLENT")
                    .when(col("Rating") >= 4.0,  "GOOD")
                    .when(col("Rating") >=3.0,   "AVERAGE")
                    .when(col("Rating").isNotNull(), "POOR")
                    .otherwise("UNRATED"))
                
# ── High Potential Product ───────────────────────────────────
# Good margin + active + in stock + good rating = worth promoting

.withColumn(
    "IsHighPotential",
    (col("GrossMarginPct") >= 30) &
    (col("IsActiveBool") == True) &
    (col("IsInStock") == True) &
    (col("Rating") >= 4.0)
)


# Metadata
        .withColumn("_silver_load_ts",  F.current_timestamp())
        .withColumn("_source",     lit("full_refresh_weekly"))
)

 # ── STEP 6: Write Silver as Delta ─────────────────────────────
(
 silver_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(SILVER_PATH)
)


 # ── Verify ────────────────────────────────────────────────────
total = silver_df.count()
active = silver_df.filter(col("IsActiveBool") == True).count()
in_stock = silver_df.filter(col("IsInStock") == True).count()
new_products = silver_df.filter(col("IsNewProduct") == "NEW").count()
high_pot = silver_df.filter(col("IsHighPotential") == True).count()  
out_of_status = silver_df.filter(col("StockStatus") == "OUT_OF_STOCK").count()



print("\n[DONE] silver/products/ written:{total} rows")
print(f" Active products: {active}")
print(f" In stock: {in_stock}")
print(f" Out of stock: {out_of_status}")
print(f" New (last 90 days): {new_products}")
print(f" High potential: {high_pot}")

print("\n[CATEGORY BREAKDOWN]")
silver_df.groupBy("Category")\
    .agg(
        F.count("ProductID").alias("products"),
        F.avg("ListPrice").alias("avg_price"),
        F.avg("GrossMarginPct").alias("avg_margin_pct"),
        F.avg("Rating").alias("avg_rating")
    )\
.withColumn("avg_price", round(col("avg_price"),0))\
.withColumn("avg_margin_pct",  round(col("avg_margin_pct"),1))\
.withColumn("avg_rating",  round(col("avg_rating"),2))\
.orderBy("products", ascending = False)\
.show()


print("[PRICE BAND BREAKDOWN]")
silver_df.groupBy("PriceBand").count()\
.orderBy("count", ascending = False)\
.show()



print("[STOCK STATUS BREAKDOWN]")
silver_df.groupBy("StockStatus").count()\
.orderBy("count", ascending = False)\
.show()



display(silver_df.select(
    "ProductID", "ProductName", "Category", "Brand",
    "ListPrice", "CostPrice", "GrossMarginPct",
    "PriceBand", "StockStatus", "RatingBand",
    "IsNewProduct", "IsHighPotential").limit(10))




[DONE] Rows read: <bound method DataFrame.count of DataFrame[ProductID: string, ProductName: string, Category: string, SubCategory: string, Brand: string, SellerID: string, ListPrice: decimal(10,2), CostPrice: decimal(10,2), StockQuantity: int, Rating: decimal(3,1), IsActive: string, LaunchDate: date, LastModifiedDate: timestamp]> rows
root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- SellerID: string (nullable = true)
 |-- ListPrice: decimal(10,2) (nullable = true)
 |-- CostPrice: decimal(10,2) (nullable = true)
 |-- StockQuantity: integer (nullable = true)
 |-- Rating: decimal(3,1) (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- LaunchDate: date (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[CLEAN] After dedup: 300 rows

[DONE] silver/products/ written:{total} rows
 Active products: 2

ProductID,ProductName,Category,Brand,ListPrice,CostPrice,GrossMarginPct,PriceBand,StockStatus,RatingBand,IsNewProduct,IsHighPotential
PROD0001,Nike Electronics Product 1,Electronics,Prestige,1301.53,987.38,24.14,MID_RANGE,IN_STOCK,GOOD,ESTABLISHED,false
PROD0002,Nike Electronics Product 2,Electronics,Penguin,12975.89,8831.65,31.94,LUXURY,IN_STOCK,AVERAGE,MATURE,false
PROD0003,Prestige Electronics Product 3,Electronics,Nike,5323.83,2695.62,49.37,PREMIUM,IN_STOCK,GOOD,ESTABLISHED,true
PROD0004,Penguin Electronics Product 4,Electronics,Samsung,3561.65,1997.49,43.92,PREMIUM,IN_STOCK,AVERAGE,ESTABLISHED,false
PROD0005,Nike Electronics Product 5,Electronics,Apple,12359.01,9630.74,22.08,LUXURY,IN_STOCK,EXCELLENT,MATURE,false
PROD0006,Prestige Electronics Product 6,Electronics,Samsung,18875.43,9579.18,49.25,LUXURY,IN_STOCK,AVERAGE,ESTABLISHED,false
PROD0007,Samsung Electronics Product 7,Electronics,Bosch,9737.53,5322.17,45.34,PREMIUM,IN_STOCK,EXCELLENT,MATURE,false
PROD0008,Penguin Electronics Product 8,Electronics,Bosch,11192.50,7446.92,33.47,LUXURY,IN_STOCK,GOOD,MATURE,true
PROD0009,Lakme Electronics Product 9,Electronics,Prestige,3177.11,2523.58,20.57,PREMIUM,IN_STOCK,POOR,MATURE,false
PROD0010,Apple Electronics Product 10,Electronics,Bosch,3952.82,2634.57,33.35,PREMIUM,MEDIUM_STOCK,AVERAGE,MATURE,false


In [0]:
display(silver_df)

ProductID,ProductName,Category,SubCategory,Brand,SellerID,ListPrice,CostPrice,StockQuantity,Rating,IsActive,LaunchDate,LastModifiedDate,IsActiveBool,GrossMarginAmt,GrossMarginPct,PriceBand,StockStatus,IsInStock,DaysSinceLaunch,IsNewProduct,RatingBand,IsHighPotential,_silver_load_ts,_source
PROD0001,Nike Electronics Product 1,Electronics,Smartphones,Prestige,SELL005,1301.53,987.38,191,4.3,FALSE,2023-12-14,2023-12-14,false,314.15,24.14,MID_RANGE,IN_STOCK,true,939,ESTABLISHED,GOOD,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0002,Nike Electronics Product 2,Electronics,Laptops,Penguin,SELL041,12975.89,8831.65,491,3.7,TRUE,2021-10-22,2021-10-22,true,4144.24,31.94,LUXURY,IN_STOCK,true,1722,MATURE,AVERAGE,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0003,Prestige Electronics Product 3,Electronics,Laptops,Nike,SELL025,5323.83,2695.62,157,4.3,TRUE,2023-11-21,2023-11-21,true,2628.21,49.37,PREMIUM,IN_STOCK,true,962,ESTABLISHED,GOOD,true,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0004,Penguin Electronics Product 4,Electronics,Cameras,Samsung,SELL012,3561.65,1997.49,308,3.3,TRUE,2023-12-08,2023-12-08,true,1564.16,43.92,PREMIUM,IN_STOCK,true,945,ESTABLISHED,AVERAGE,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0005,Nike Electronics Product 5,Electronics,Tablets,Apple,SELL016,12359.01,9630.74,177,5.0,TRUE,2022-10-13,2022-10-13,true,2728.27,22.08,LUXURY,IN_STOCK,true,1366,MATURE,EXCELLENT,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0006,Prestige Electronics Product 6,Electronics,Headphones,Samsung,SELL017,18875.43,9579.18,107,3.1,TRUE,2023-12-22,2023-12-22,true,9296.25,49.25,LUXURY,IN_STOCK,true,931,ESTABLISHED,AVERAGE,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0007,Samsung Electronics Product 7,Electronics,Headphones,Bosch,SELL047,9737.53,5322.17,214,4.9,FALSE,2020-01-03,2020-01-03,false,4415.36,45.34,PREMIUM,IN_STOCK,true,2380,MATURE,EXCELLENT,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0008,Penguin Electronics Product 8,Electronics,Smartphones,Bosch,SELL037,11192.50,7446.92,333,4.4,TRUE,2023-01-23,2023-01-23,true,3745.58,33.47,LUXURY,IN_STOCK,true,1264,MATURE,GOOD,true,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0009,Lakme Electronics Product 9,Electronics,Smartphones,Prestige,SELL017,3177.11,2523.58,487,2.8,TRUE,2020-03-09,2020-03-09,true,653.53,20.57,PREMIUM,IN_STOCK,true,2314,MATURE,POOR,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
PROD0010,Apple Electronics Product 10,Electronics,Tablets,Bosch,SELL046,3952.82,2634.57,15,3.6,TRUE,2020-06-02,2020-06-02,true,1318.25,33.35,PREMIUM,MEDIUM_STOCK,true,2229,MATURE,AVERAGE,false,2026-07-10T21:06:55.283172Z,full_refresh_weekly
